# Chapter 15: Evaluating Agents Properly
## Topic 1: Why Evaluating an Agent Is Harder Than a Classifier

**One notebook. Theory + Code together.**


## Part A: Theory

### 1. Concept, Intuition, and Why This Exists

- Chapter 1's original classifier evaluation was conceptually simple: one email in, one label out, compared against one ground-truth label — accuracy, precision, recall are all well-defined, single-number-per-example comparisons. Every agent built since Chapter 3 — with tool calls (Chapter 10), memory (Chapter 11), retrieval decisions (Chapter 13) — breaks this simplicity in a specific, structural way this topic names directly: an agent doesn't produce one output for one input, it produces a **sequence of decisions**, any one of which could be wrong, even when the final answer happens to look correct.
- The core structural difference this topic centers on: a classifier's evaluation question is "was the single output correct?" — a well-posed question with an unambiguous answer. An agent's evaluation question is actually several different, only loosely related questions layered on top of each other: did it call the right tools, in the right order, with the right arguments, did it correctly interpret each tool's result, and did all of that ultimately produce the right final answer? A classifier evaluation collapses to one comparison; an agent evaluation is inherently a **multi-step process evaluation**, and this topic exists to make that structural difference explicit before diving into how to actually measure it (Topics 2-5).
- Why this matters concretely for this project: Chapter 10 Topic 7's own test-suite work already implicitly encountered this distinction (separating triggering/selection tests from argument-correctness tests from result-handling tests) without naming the underlying principle directly — this topic is where that implicit structure gets its explicit name and justification.


### 2. Internal Working — Step by Step

**The specific ways agent evaluation genuinely differs from classifier evaluation, one at a time:**

1. **A correct final answer doesn't guarantee a correct process** — an agent could call the wrong tool, receive an irrelevant result, and still stumble into a correct-sounding final answer by coincidence (or vice versa: call exactly the right tools correctly, and still produce a wrong final answer due to a generation-layer mistake). A classifier has no equivalent "right answer, wrong process" failure mode, since there's no multi-step process to get right or wrong in the first place — only Chapter 8 Topic 3's faithfulness/relevance/correctness distinction hints at something similar, and this topic generalizes that same insight to the full agent, not just the generation step.
2. **The "correct" trajectory is often not unique** — for many real queries, several different tool-calling sequences could all legitimately lead to a correct final answer (calling `search_knowledge_base` before or after `check_sender_history`, for instance, might both be reasonable). A classifier's ground truth is a single correct label; an agent's ground truth for "did it behave correctly" may need to accommodate multiple acceptable trajectories, not just one.
3. **Partial correctness is a genuinely different kind of partial than a classifier's confusion matrix represents** — a classifier's wrong prediction is simply wrong; an agent that correctly retrieves the right information but generates a slightly incomplete final answer, or that calls the right tool with a slightly malformed argument that still happens to work, sits somewhere between fully correct and fully incorrect in a way a single label comparison cannot represent at all.
4. **Evaluation itself requires deciding what counts as an evaluable "example" in the first place** — a classifier's evaluation set is simply a list of (input, correct label) pairs. An agent's evaluation set needs, at minimum, an expected final answer, but genuinely rigorous evaluation (Topics 2-4) also needs some notion of expected tool calls, expected tool-call order or acceptable orderings, and expected tool arguments — a considerably richer, more expensive-to-construct evaluation artifact than a classifier ever required.


### 3. How This Is Implemented in This Project

- Chapter 1's classifier evaluation (`classify_by_keyword_rules()`, and later the ML/GenAI layers built on top of it) fits the simple, single-comparison evaluation model this topic contrasts against — one email, one label, one comparison against ground truth.
- Every agent-evaluation exercise already built in this notebook series — Chapter 10 Topic 7's three-category test suite (triggering, argument-correctness, result-handling), Chapter 13 Topic 4's three-layer Smart Saver FD test (triggering decision, downstream grounding, full-chain correctness) — was, in each case, already implicitly grappling with this topic's core insight: no single pass/fail check adequately captures whether the agent behaved correctly, and each of those exercises independently arrived at a layered, multi-dimensional evaluation approach rather than a single comparison.
- This project's agent (`run_agent()`, extended with memory in Chapter 11 and retrieval-triggering logic in Chapter 13) is exactly the kind of system this topic's distinction applies to directly — evaluating whether it "worked correctly" for a given email is a genuinely richer question than evaluating whether Chapter 1's simpler classifier assigned the correct FD/Non-FD/Multiple-Category label.


### 4. Real-World Issues, Edge Cases, Debugging, Monitoring, Scaling, Latency, Cost, Security, Deployment

- **Treating agent evaluation as if it were classifier evaluation (checking only the final output) is a real, common, and costly mistake** — it silently misses process failures a correct-looking final answer can mask entirely, exactly the "right answer, wrong process, got lucky" risk this topic's first point describes. A team relying solely on final-answer accuracy could ship an agent with a systematically broken tool-selection process that happens to produce acceptable answers on the specific evaluation set used, while failing unpredictably on slightly different real queries the eval set didn't happen to cover.
- **Constructing a genuinely rigorous agent evaluation set is considerably more expensive than constructing a classifier evaluation set** — Chapter 14 Topic 2's own real, hands-on eval-set-construction exercise already demonstrated some of this expense (extracting questions, writing verified reference answers); a full agent evaluation set additionally needs expected-tool-call information, multiplying the labeling effort further, and this cost needs to be budgeted for realistically, not assumed comparable to classifier-evaluation-set construction cost.
- **The "multiple acceptable trajectories" problem creates genuine ambiguity in what counts as a test failure** — if an agent takes a legitimate-but-different path to the same correct final answer, is that a pass or a labeling gap? This needs an explicit, deliberate evaluation design decision (Topics 2-4 will address this directly), not an implicit assumption that only one "correct" trajectory exists per test case.
- **Debugging an agent evaluation failure requires knowing which specific step failed**, exactly the layered-attribution discipline this notebook series has repeatedly required (Chapter 10 Topic 1's three-category failure distinction, Chapter 13 Topic 4's three-layer test) — a coarse, final-answer-only evaluation provides none of this diagnostic value, forcing manual investigation for every single failure rather than getting specific, actionable information directly from the evaluation itself.
- **Monitoring an agent in production requires tracking process-level signals**, not just final-output correctness — exactly what Chapter 14 Topic 4's tracing infrastructure was built to support, and what Chapter 10 Topic 6's tool-combination monitoring already anticipated, both existing specifically because final-output-only monitoring would miss exactly the process failures this topic identifies as uniquely characteristic of agent systems.


### 5. Design Decisions, Trade-offs, and Real-Time Dilemmas

- **Final-answer-only evaluation vs full-process evaluation:** final-answer-only evaluation is far cheaper to construct and run, but silently misses process failures a correct-looking answer can mask — the right choice only for a very early-stage, low-stakes prototype, not for any system where process correctness genuinely matters (which describes most production agent systems, including this project's own regulated-domain context).
- **Requiring one single "correct" trajectory per test case vs allowing multiple acceptable trajectories:** requiring a single correct trajectory is simpler to construct and score but risks incorrectly failing an agent that took a legitimate alternative path — allowing multiple acceptable trajectories is more accurate but requires more upfront thought about what actually counts as acceptable for each specific test case, a genuine additional labeling cost that needs to be weighed against the risk of false failures.
- **How much of this richer evaluation to build immediately vs incrementally:** given the real, additional cost of constructing rigorous agent evaluation data (expected tool calls, acceptable trajectory variations), a reasonable approach builds this incrementally — starting with final-answer checks (cheap, immediate) and layering in tool-call-level checks (Topics 2-3) as specific process failures are actually observed and need targeted evaluation coverage, rather than attempting full, comprehensive process evaluation from the very first day.


### 6. Alternatives and When to Use Each

- **Classifier-style, final-output-only evaluation:** appropriate specifically for evaluating this project's non-agentic components — Chapter 1's rule-based classifier, or any single-step, non-tool-calling model call — where there genuinely is no multi-step process to evaluate.
- **Full, multi-dimensional agent evaluation (this chapter's overall subject, built out across Topics 2-5):** the right approach for any genuinely agentic component of this project — the tool-calling, memory-using, retrieval-deciding agent this notebook series has built since Chapter 3.
- **A hybrid approach — final-answer checks for broad coverage, supplemented by targeted process-level checks for specific, previously-observed failure modes:** a pragmatic middle ground worth adopting incrementally, exactly mirroring how Chapter 10 Topic 7's test suite grew from illustrative single-example checks into a more structured, multi-category practice over time.


### 7. Common Mistakes and Production Failures

- Evaluating an agent using only final-answer accuracy, silently missing process failures a correct-looking output can mask entirely — exactly the "right answer for the wrong reason" risk this topic identifies as uniquely characteristic of multi-step agent systems.
- Assuming a single "correct" trajectory exists per test case when multiple legitimate paths could reach the same correct answer, incorrectly failing an agent for taking a valid alternative approach.
- Underestimating the real cost of constructing a rigorous agent evaluation set (which needs expected tool calls in addition to expected final answers), leading to under-resourced, insufficiently rigorous evaluation infrastructure.
- Not budgeting for the genuinely higher complexity of debugging an agent evaluation failure compared to a classifier evaluation failure, given the layered-attribution work multi-step failures require.
- Treating agent evaluation and classifier evaluation as needing the same tooling and process, rather than recognizing the structural difference this topic identifies and building appropriately different evaluation infrastructure for each.


### 8. Lead-Level Interview Questions

**Basic**

- Q: What is the core structural reason agent evaluation is harder than classifier evaluation?
  A: A classifier produces one output for one input, evaluated by a single, unambiguous comparison against ground truth. An agent produces a sequence of decisions — which tools to call, in what order, with what arguments, how to interpret each result — any one of which could be wrong even when the final answer happens to look correct. Agent evaluation is inherently a multi-step process evaluation, not a single-output comparison.

- Q: Why can a correct final answer still represent an agent evaluation failure?
  A: The agent could have followed an incorrect or unreliable process — calling the wrong tool, misinterpreting a result — and still arrived at a correct-looking final answer by coincidence. This "right answer, wrong process" pattern has no equivalent in classifier evaluation, since there's no multi-step process involved that could be right or wrong independent of the final output.

**Intermediate**

- Q: Why does the existence of "multiple acceptable trajectories" complicate agent evaluation in a way classifier evaluation never faces?
  A: A classifier's ground truth is a single correct label — there's no ambiguity about what the right answer is. An agent, for many real queries, might have several different legitimate tool-calling sequences that all correctly lead to the same right final answer, meaning agent evaluation needs an explicit design decision about whether to require one specific expected trajectory or accommodate multiple acceptable ones — a genuinely new kind of evaluation-design problem classifier evaluation never has to solve.

- Q: How does constructing an agent evaluation set differ in cost and complexity from constructing a classifier evaluation set?
  A: A classifier evaluation set only needs (input, correct label) pairs. A rigorous agent evaluation set additionally needs expected tool calls (which tools, in what order or acceptable orderings, with what arguments) for each test case — a considerably richer, more expensive-to-construct artifact, since it requires domain expertise not just about the correct final answer, but about the correct or acceptable process for reaching it.

**Advanced**

- Q: Design a case where a classifier-style, final-answer-only evaluation would give a false sense of confidence about an agent's actual reliability.
  A: Consider an agent that, for a specific query type, sometimes calls `validate_fd_reference` with a slightly malformed reference number that happens to coincidentally still match a real record (due to some fuzzy matching quirk), producing a correct-looking final answer despite this process flaw. Final-answer-only evaluation would score this as a pass, hiding the fact that this same process flaw might, on a different email with a different malformed reference number, retrieve a completely different customer's record — a serious, silent risk (directly connecting to Chapter 9 Topic 4's exact-match-only principle) that only tool-call-level evaluation (Topic 3) would actually catch.

- Q: A teammate argues that since the agent's final answers are consistently correct on your evaluation set, process-level evaluation is unnecessary overhead. How do you respond?
  A: Consistently correct final answers on a specific evaluation set don't guarantee the underlying process generalizes correctly to different, unseen queries — the "right answer, wrong process, got lucky" pattern is specifically dangerous because it can persist unnoticed until a slightly different real-world case exposes it. Process-level evaluation isn't overhead; it's what actually validates that the agent's behavior is reliable and generalizable, not just that it happened to produce acceptable answers for the specific cases already tested — exactly the distinction this topic's entire argument rests on.

**Scenario-based**

- Q: Your agent's final-answer accuracy on the evaluation set is 95%, but a new production incident report suggests the agent is behaving inconsistently for a query type not well-represented in that evaluation set. How does this topic's framework inform your response?
  A: This is a direct illustration of the topic's core warning — high final-answer accuracy on an existing evaluation set says nothing about process reliability for query types the set doesn't adequately cover. The appropriate response is expanding the evaluation set to include this specific, previously-underrepresented query type, and specifically adding process-level checks (Topics 2-3) for it, rather than trusting the existing 95% final-answer accuracy figure as sufficient evidence the agent is behaving reliably overall.

**Follow-up questions to expect:**

- "How would you decide how much process-level evaluation detail is worth the additional construction cost for a specific project?"
- "What would you do if two team members disagreed about whether a specific agent trajectory should count as 'acceptable' for a given test case?"


### 9. Hidden Concepts / Prerequisites Worth Knowing

- **This topic's core insight — that process correctness and outcome correctness are genuinely separable properties — is a specific instance of a much more general evaluation principle** appearing throughout computer science and beyond: verifying that a system reached the right answer is often a fundamentally different and sometimes easier problem than verifying that it reached that answer through a reliable, generalizable process, a distinction with echoes in software testing, scientific methodology, and decision-making evaluation generally.
- **The "multiple acceptable trajectories" problem is conceptually related to non-unique-solution-path problems studied more formally in planning and search algorithms** — recognizing that a "correct" sequence of actions isn't always unique is a foundational idea in classical AI planning, now showing up here in the specific, applied context of evaluating an LLM-based agent's tool-use behavior.
- **This topic is the conceptual foundation for the rest of Chapter 15**: Topic 2's task-level vs step-level metrics distinction, Topic 3's tool-call-correctness checking, and Topics 4-5's harness-and-regression-tracking infrastructure are all direct, practical responses to the structural problem this topic identifies — richer evaluation because agent behavior is structurally richer than a classifier's single-output behavior.

### 10. Quick Revision Summary (for last-minute interview prep)

> Evaluating an agent is structurally harder than evaluating a classifier because an agent produces a sequence of decisions — tool selection, tool-call arguments, result interpretation, final generation — any of which can be wrong even when the final output happens to look correct, a "right answer, wrong process" failure mode with no equivalent in single-output classifier evaluation. This is compounded by the fact that a "correct" trajectory is often not unique — several different legitimate tool-calling sequences might all lead to the same correct final answer, requiring an explicit evaluation-design decision about whether to require one specific expected trajectory or accommodate multiple acceptable ones. Constructing a rigorous agent evaluation set is genuinely more expensive than constructing a classifier evaluation set, since it needs expected tool-call information in addition to expected final answers. This project's own earlier evaluation exercises (Chapter 10 Topic 7's three-category test suite, Chapter 13 Topic 4's three-layer Smart Saver FD test) already implicitly grappled with this exact structural distinction — this topic makes that underlying principle explicit, setting up the rest of this chapter's task-level vs step-level metrics (Topic 2), tool-call correctness (Topic 3), and repeatable, regression-tracking evaluation infrastructure (Topics 4-5) as direct, necessary responses to agent evaluation's genuinely richer structure.


### Module 1: A Real Agent That Sometimes Gets the Right Answer Through the Wrong Process

Build a simulated agent with a real, injectable process flaw (a fuzzy-matching bug in tool argument construction), and show it can still produce a correct-looking final answer.

In [1]:

# ------------------------------------------------------------------
# MODULE 1: An agent with a REAL, injectable process flaw
# ------------------------------------------------------------------

FD_RECORDS = {
    "BJ2019FD7717": {"customer_name": "Shobha Chopra", "amount": 391000},
    "BJ2019FD7718": {"customer_name": "Rajesh Kumar", "amount": 275000},  # note: differs by ONE digit
}

def validate_fd_reference_exact(reference_number: str) -> dict:
    """CORRECT tool implementation -- exact match only, per Chapter 9
    Topic 4's established principle."""
    record = FD_RECORDS.get(reference_number)
    if record is None:
        return {"found": False}
    return {"found": True, **record}


def validate_fd_reference_FLAWED(reference_number: str) -> dict:
    """A FLAWED tool implementation with an injected process bug --
    if the exact reference isn't found, it falls back to a 'fuzzy'
    match on the first 10 characters. This is a REAL process flaw,
    not just a hypothetical one."""
    if reference_number in FD_RECORDS:
        return {"found": True, **FD_RECORDS[reference_number]}
    # FLAWED fallback: fuzzy match on a prefix
    for ref, record in FD_RECORDS.items():
        if ref[:10] == reference_number[:10]:
            return {"found": True, **record}  # WRONG match, but looks "found"
    return {"found": False}


def agent_answer_query(reference_number: str, tool_function) -> dict:
    """A simplified agent: calls the given tool function, then
    constructs a final answer from whatever it returns."""
    result = tool_function(reference_number)
    if result["found"]:
        answer = f"Your FD amount is Rs {result['amount']:,}, registered to {result['customer_name']}."
    else:
        answer = "We could not find that FD reference number."
    return {"tool_result": result, "answer": answer}


print("=" * 70)
print("MODULE 1: CORRECT vs FLAWED TOOL IMPLEMENTATION")
print("=" * 70)

# a customer asks about THEIR OWN correct reference number
query_ref = "BJ2019FD7717"

correct_result = agent_answer_query(query_ref, validate_fd_reference_exact)
print(f"\nQuery: '{query_ref}' (genuinely exists)")
correct_answer_1 = correct_result["answer"]
print(f"  CORRECT tool -> answer: '{correct_answer_1}'")

flawed_result = agent_answer_query(query_ref, validate_fd_reference_FLAWED)
flawed_answer_1 = flawed_result["answer"]
print(f"  FLAWED tool  -> answer: '{flawed_answer_1}'")
print(f"  (Same answer here, since the exact reference genuinely exists --")
print(f"   the flaw is NOT yet visible from this query alone.)")

print("\nModule 1 complete. Run Module 2 next.")


MODULE 1: CORRECT vs FLAWED TOOL IMPLEMENTATION

Query: 'BJ2019FD7717' (genuinely exists)
  CORRECT tool -> answer: 'Your FD amount is Rs 391,000, registered to Shobha Chopra.'
  FLAWED tool  -> answer: 'Your FD amount is Rs 391,000, registered to Shobha Chopra.'
  (Same answer here, since the exact reference genuinely exists --
   the flaw is NOT yet visible from this query alone.)

Module 1 complete. Run Module 2 next.


### Module 2: The Flaw Surfaces — A Different Customer's Typo'd Reference

Show the flawed tool producing a confidently WRONG but correct-LOOKING answer for a different, slightly-mistyped reference number -- exactly the right-answer-wrong-process risk this topic describes.

In [2]:

# ------------------------------------------------------------------
# MODULE 2: The flaw surfaces on a DIFFERENT, typo'd query
# ------------------------------------------------------------------

# a DIFFERENT customer, with a genuine, real typo in their own reference number
typo_ref = "BJ2019FD771X"  # customer meant BJ2019FD7717, mistyped last character

print("=" * 70)
print("MODULE 2: THE PROCESS FLAW SURFACES ON A TYPO'D REFERENCE")
print("=" * 70)
print(f"\nQuery: '{typo_ref}' (a TYPO -- does not exactly match any real record)")

correct_result_2 = agent_answer_query(typo_ref, validate_fd_reference_exact)
correct_answer_2 = correct_result_2["answer"]
print(f"\n  CORRECT tool -> answer: '{correct_answer_2}'")
print(f"  (Correctly reports NOT FOUND -- honest, appropriate behavior for a typo)")

flawed_result_2 = agent_answer_query(typo_ref, validate_fd_reference_FLAWED)
flawed_answer_2 = flawed_result_2["answer"]
print(f"\n  FLAWED tool  -> answer: '{flawed_answer_2}'")
print(f"  (WRONG -- confidently returned Shobha Chopra's Rs 391,000 record for")
print(f"   a query that should have found NOTHING. This customer would see")
print(f"   ANOTHER PERSON'S account details -- a real, serious risk.)")

print("\nCONFIRMED: the flawed tool's process bug is INVISIBLE when tested")
print("only against genuinely-existing reference numbers (Module 1), but")
print("produces a confidently WRONG, cross-customer-data-leaking answer")
print("for a realistic typo -- exactly the kind of process failure a")
print("final-answer-only evaluation, run only against clean test cases,")
print("would never catch.")

print("\nModule 2 complete. Run Module 3 next.")


MODULE 2: THE PROCESS FLAW SURFACES ON A TYPO'D REFERENCE

Query: 'BJ2019FD771X' (a TYPO -- does not exactly match any real record)

  CORRECT tool -> answer: 'We could not find that FD reference number.'
  (Correctly reports NOT FOUND -- honest, appropriate behavior for a typo)

  FLAWED tool  -> answer: 'Your FD amount is Rs 391,000, registered to Shobha Chopra.'
  (WRONG -- confidently returned Shobha Chopra's Rs 391,000 record for
   a query that should have found NOTHING. This customer would see
   ANOTHER PERSON'S account details -- a real, serious risk.)

CONFIRMED: the flawed tool's process bug is INVISIBLE when tested
only against genuinely-existing reference numbers (Module 1), but
produces a confidently WRONG, cross-customer-data-leaking answer
for a realistic typo -- exactly the kind of process failure a
final-answer-only evaluation, run only against clean test cases,
would never catch.

Module 2 complete. Run Module 3 next.


### Module 3: Final-Answer-Only Evaluation vs Process-Aware Evaluation, Measured

Run BOTH evaluation styles against the same test set, showing that final-answer-only evaluation gives the flawed agent a passing grade, while process-aware evaluation correctly catches the flaw.

In [3]:

# ------------------------------------------------------------------
# MODULE 3: Final-answer-only vs process-aware evaluation, measured
# ------------------------------------------------------------------

TEST_CASES = [
    {"reference": "BJ2019FD7717", "expected_found": True, "expected_customer": "Shobha Chopra"},
    {"reference": "BJ2019FD7718", "expected_found": True, "expected_customer": "Rajesh Kumar"},
    {"reference": "BJ2019FD771X", "expected_found": False, "expected_customer": None},  # the typo case
]


def evaluate_final_answer_only(test_cases: list, tool_function) -> dict:
    """CLASSIFIER-STYLE evaluation: only checks whether SOME answer was
    produced, not whether the underlying tool_result was actually correct."""
    passed = 0
    for case in test_cases:
        result = agent_answer_query(case["reference"], tool_function)
        # a naive final-answer check: did we get ANY non-empty answer?
        got_an_answer = len(result["answer"]) > 0
        if got_an_answer:
            passed += 1
    return {"passed": passed, "total": len(test_cases)}


def evaluate_process_aware(test_cases: list, tool_function) -> dict:
    """PROCESS-AWARE evaluation: checks whether the tool_result's
    found/not-found status AND the specific customer match what was
    EXPECTED -- catching the process flaw directly."""
    passed = 0
    failures = []
    for case in test_cases:
        result = agent_answer_query(case["reference"], tool_function)
        tool_result = result["tool_result"]
        found_correct = tool_result["found"] == case["expected_found"]
        customer_correct = (not case["expected_found"]) or (
            tool_result.get("customer_name") == case["expected_customer"]
        )
        is_correct = found_correct and customer_correct
        if is_correct:
            passed += 1
        else:
            failures.append(case["reference"])
    return {"passed": passed, "total": len(test_cases), "failures": failures}


print("=" * 70)
print("FINAL-ANSWER-ONLY vs PROCESS-AWARE EVALUATION -- MEASURED")
print("=" * 70)

print("\n[FLAWED TOOL, evaluated FINAL-ANSWER-ONLY style]")
final_only_result = evaluate_final_answer_only(TEST_CASES, validate_fd_reference_FLAWED)
fo_passed = final_only_result["passed"]
fo_total = final_only_result["total"]
print(f"  Passed: {fo_passed}/{fo_total}")

print("\n[FLAWED TOOL, evaluated PROCESS-AWARE style]")
process_aware_result = evaluate_process_aware(TEST_CASES, validate_fd_reference_FLAWED)
pa_passed = process_aware_result["passed"]
pa_total = process_aware_result["total"]
print(f"  Passed: {pa_passed}/{pa_total}")
pa_failures = process_aware_result["failures"]
print(f"  Failed cases: {pa_failures}")

if final_only_result["passed"] == final_only_result["total"] and process_aware_result["passed"] < process_aware_result["total"]:
    print("\nCONFIRMED: final-answer-only evaluation gave the FLAWED tool a")
    print(f"PERFECT {fo_passed}/{fo_total} score -- completely missing the real,")
    print("serious cross-customer data leak risk. Process-aware evaluation")
    pa_failed_count = pa_total - pa_passed
    print(f"correctly caught it, failing {pa_failed_count} of {pa_total} cases -- exactly")
    print("this topic's core argument, demonstrated with real, executable code.")

print("\nModule 3 complete. All Chapter 15 Topic 1 modules done.")
print()
print("=" * 70)
print("CHAPTER 15 TOPIC 1 -- KEY POINTS TO REMEMBER")
print("=" * 70)
print("""
  "Right answer, wrong process" is a REAL, demonstrable failure mode --
  a flawed tool implementation produced CORRECT-LOOKING answers for
  genuinely-existing references, hiding a serious cross-customer data
  leak risk that only surfaced on a realistic typo'd reference.

  Final-answer-only evaluation gave the FLAWED tool a PERFECT score --
  demonstrated concretely with real, measured evaluation results.

  Process-aware evaluation (checking the tool's ACTUAL result, not just
  whether SOME answer was produced) correctly caught the flaw.

  This is exactly why agent evaluation needs to check the PROCESS
  (which tool, what result, was it actually correct), not just the
  final output -- a classifier has no equivalent process to check,
  but an agent's tool-calling behavior absolutely does.
""")


FINAL-ANSWER-ONLY vs PROCESS-AWARE EVALUATION -- MEASURED

[FLAWED TOOL, evaluated FINAL-ANSWER-ONLY style]
  Passed: 3/3

[FLAWED TOOL, evaluated PROCESS-AWARE style]
  Passed: 2/3
  Failed cases: ['BJ2019FD771X']

CONFIRMED: final-answer-only evaluation gave the FLAWED tool a
PERFECT 3/3 score -- completely missing the real,
serious cross-customer data leak risk. Process-aware evaluation
correctly caught it, failing 1 of 3 cases -- exactly
this topic's core argument, demonstrated with real, executable code.

Module 3 complete. All Chapter 15 Topic 1 modules done.

CHAPTER 15 TOPIC 1 -- KEY POINTS TO REMEMBER

  "Right answer, wrong process" is a REAL, demonstrable failure mode --
  a flawed tool implementation produced CORRECT-LOOKING answers for
  genuinely-existing references, hiding a serious cross-customer data
  leak risk that only surfaced on a realistic typo'd reference.

  Final-answer-only evaluation gave the FLAWED tool a PERFECT score --
  demonstrated concretely with real